# Time Series Similarity Search — Practice lab

Data Acquisition, Extraction and Storage — M2 (PSL)

**Name(s):** ...

**Personal seed $u_0$:** ...

### How to use this notebook
* The subject (PDF) contains the definitions and the full statements. This notebook follows its structure.
* **Questions Q** (numerical): compute the four items a)–d) and store them in the variables `Q1`, `Q2`, ... (lists of 4 values). They are gathered in the dictionary `ANSWERS` at the end of the notebook.
* **Questions T** (theory / experiments): answer in the Markdown cells marked *Your answer*. Proofs, short algorithm descriptions, complexities, plots + comments. Do not paste your code there.
* **Allowed:** Python standard library, `numpy`, `matplotlib`. **Not allowed:** any library computing the requested distances / representations / indexes for you (`scipy.spatial`, `sklearn.neighbors`, `tslearn`, `sktime`, `aeon`, `pyts`, `dtaidistance`, `fastdtw`, `stumpy`, `rtree`, `faiss`, ...).
* **Work first with `SEED = 42`** and compare with the reference sheet (the function `check_reference()` at the end does it for you). Then set `SEED` to your personal seed, re-run everything, and keep the notebook in that state.
* Real-valued answers: rounded to 2 decimals. Counting questions: strict inequalities.

In [4]:
import math, time, heapq, bisect, itertools
from collections import deque
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import unicodedata

def seed_from_name(family_name: str) -> int:
    """Deterministic personal seed in [1, 10^6] computed from a family name (DO NOT MODIFY)."""
    s = unicodedata.normalize("NFKD", family_name)
    s = "".join(c for c in s if c.isalpha() and not unicodedata.combining(c)).lower()
    h = int.from_bytes(hashlib.sha256(s.encode("utf-8")).digest()[:8], "big")
    seed = 1 + h % 1_000_000
    return seed if seed != 42 else 43          # 42 is reserved for the reference sheet


#SEED = seed_from_name("YOUR-LAST-NAME")
SEED = 42          # <-- replace by your personal seed once your code is validated



## 1. Preliminaries — the dataset
The cell below is the generator (Appendix A of the subject). **Do not modify it.**

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# Pseudo-random generator: DO NOT MODIFY (it defines your personal dataset)
# ---------------------------------------------------------------------------
MOD = 2 ** 31
A, C = 1103515245, 12345


class LCG:
    """Tiny linear congruential generator, identical on every platform."""

    def __init__(self, seed):
        self.state = seed % MOD
        for _ in range(8):                      # warm-up
            self.next()

    def next(self):
        self.state = (A * self.state + C) % MOD
        return self.state

    def randint(self, a, b):
        """Uniform integer in [a, b] (both included)."""
        return a + (self.next() >> 16) % (b - a + 1)


def _rng(seed, tag, k):
    return LCG(seed * 1_000_003 + tag * 7_919 + k * 104_729 + 1)


def _walk(rng, l):
    """Integer-valued signal: slow random walk + 1 to 3 rectangular bumps."""
    x = rng.randint(-10, 10)
    t = [0] * l
    for i in range(l):
        t[i] = x
        if rng.randint(0, 3) == 0:
            x += rng.randint(-1, 1) * 2
    for _ in range(rng.randint(1, 3)):
        h = rng.randint(8, 15) * (1 if rng.randint(0, 1) else -1)
        w = rng.randint(max(1, l // 32), max(1, l // 16))
        p = rng.randint(0, l - w)
        for i in range(p, p + w):
            t[i] += h
    return t


def series(seed, k, l):
    """The series T_k of length l (a numpy array of integers)."""
    return np.array(_walk(_rng(seed, 0, k), l), dtype=np.int64)


def database(seed, n=1000, l=256):
    """The database D = [T_0, ..., T_{n-1}] as an (n, l) numpy array of integers."""
    return np.stack([series(seed, k, l) for k in range(n)])


QUERY_TYPES = [(0, 1), (0, 4), (10, 1), None]   # (max. warping, noise); None = new series


def query(seed, j, D):
    """The query Q_j (j in {0, 1, 2, 3}), of the same length as the series of D."""
    n, l = D.shape
    rng = _rng(seed, 1, j)
    qt = QUERY_TYPES[j % 4]
    if qt is None:
        return np.array(_walk(rng, l), dtype=np.int64)
    warp, noise = qt
    src = D[rng.randint(0, n - 1)]
    s, mode, q = 0, 1, [0] * l
    for i in range(l):
        if i > 0:
            if rng.randint(0, 7) == 0:          # change of warping regime
                mode = rng.randint(0, 2)        # 0: stretch, 1: none, 2: compress
            step = mode if rng.randint(0, 1) == 0 else 1
            s = min(max(s + step, i - warp), i + warp, s + 2, l - 1)
        q[i] = int(src[s]) + rng.randint(-noise, noise)
    return np.array(q, dtype=np.int64)

In [ ]:
D = database(SEED)                      # D[k] is T_k, shape (1000, 256)
n, l = D.shape
Qs = [query(SEED, j, D) for j in range(4)]

fig, ax = plt.subplots(2, 4, figsize=(16, 4), sharex=True)
for j in range(4):
    ax[0, j].plot(D[j]); ax[0, j].set_title(f"T_{j}")
    ax[1, j].plot(Qs[j], c="C3"); ax[1, j].set_title(f"Q_{j}")
plt.tight_layout(); plt.show()

**Q1** — amplitude $\max_i t_i - \min_i t_i$ of $T_0, T_1, T_{500}, T_{999}$.

In [ ]:
Q1 = [None, None, None, None]
print("Q1", Q1)

---
## 2. Euclidean distance, PAA and lower-bounding search *(in class)*

In [ ]:
def ed(a, b):
    """Euclidean distance between two series of the same length."""
    raise NotImplementedError

Q2 = [None, None, None, None]     # (T0,T1), (T10,T20), (T100,T200), (T998,T999)
print("Q2", Q2)

In [ ]:
def paa(T, N):
    """PAA of order N of T (N divides len(T))."""
    raise NotImplementedError

def dr(Ta_paa, Tb_paa, l):
    """Distance DR between two PAA representations of series of length l."""
    raise NotImplementedError

Q3 = [None, None, None, None]     # DR(T0~, T1~) for N = 4, 16, 64, 256
print("Q3", Q3)

**T1** — Prove that $\mathsf{DR}\le\mathsf{ED}$.

*Your answer:*

**T2** — Compare $\mathsf{DR}$ with orders $N \mid N'$. What is $\mathsf{DR}$ when $N=l$?

*Your answer:*

In [ ]:
# Q4: brute-force nearest neighbour under ED -> list of (k, distance)
Q4 = [None, None, None, None]     # Q_0, Q_1, Q_2, Q_3
print("Q4", Q4)

**T3** — Algorithm `findNN` using the PAA (best case $O(nN+l)$), correctness, worst-case complexity, when does the best case happen?

*Your answer:*

In [ ]:
def find_nn(Q, D, D_paa, N):
    """Exact nearest neighbour of Q under ED using the PAA lower bound.
    Suggested return value: (index, distance, number of exact ED computations)."""
    raise NotImplementedError

Q5 = [None, None, None, None]     # #{T != NN : DR(Q~, T~) < ED(Q, NN)} for (Q_2, N=4), (Q_3, N=2), (Q_3, N=4), (Q_3, N=8)
print("Q5", Q5)

**T4** *(experiment)* — Link between Q5 and the number of exact computations of `findNN`; plots of the number of exact ED computations and of the running time as functions of $N\in\{2,4,\dots,128\}$; comparison with brute force; comment.

In [ ]:
# T4: your experiments here

*Your answer:*

---
## 3. Dynamic Time Warping *(in class)*

**T5** — Worst-case complexity of the trivial algorithm enumerating all admissible paths.

*Your answer:*

**T6** — A polynomial algorithm for DTW: description, correctness, time and memory complexities. Can the memory be reduced if only the distance is needed?

*Your answer:*

In [ ]:
def dtw(a, b):
    """DTW distance between two series of any lengths."""
    raise NotImplementedError

# pairs: (T_1^(50), T_2^(60)), (T_10^(300), T_20^(200)), (T_30^(1000), T_40^(1500)), (T_50^(3000), T_60^(3000))
# T_k^(l) is series(SEED, k, l)
Q6 = [None, None, None, None]
print("Q6", Q6)

**T7** — Is DTW a metric? Counter-examples, and why it matters for similarity search.

*Your answer:*

**T8** — A lower bound of DTW (any lengths) computable in $O(1)$; proof.

*Your answer:*

In [ ]:
# T8: implement your lower bound and check it empirically against dtw on random examples

**T9** — $\mathsf{DTW}_0$, $\mathsf{DTW}_{l-1}$, monotonicity in $w$, ordering between ED, $\mathsf{DTW}_w$ and DTW, complexities of your $\mathsf{DTW}_w$.

*Your answer:*

In [ ]:
def dtw_w(a, b, w):
    """DTW constrained by a Sakoe-Chiba band of width w (series of the same length)."""
    raise NotImplementedError

# (T0, T1, w=0), (T0, T1, w=10), (T0, T1, w=255), (T_5^(1e5), T_6^(1e5), w=20)
Q7 = [None, None, None, None]
print("Q7", Q7)

In [ ]:
# Q8: nearest neighbour under DTW_w -> list of (k, distance) for (Q_1, w=10), (Q_2, w=10), (Q_3, w=10), (Q_2, w=5)
Q8 = [None, None, None, None]
print("Q8", Q8)

**T10** *(experiment)* — For the four queries ($w=10$): NN under ED (Q4) vs under $\mathsf{DTW}_w$; ratio 2nd-NN / NN distance under ED and under $\mathsf{DTW}_w$; interpretation (look at how the queries are generated). What changes for $Q_2$ with $w=5$?

In [ ]:
# T10: your experiments here

*Your answer:*

---
## 4. Adaptive Piecewise Constant Approximation *(at home)*

**T11** — $\mathsf{Err}(s,T)$ in $O(1)$ after $O(l)$ pre-processing; numerical issues and how to avoid them.

*Your answer:*

In [ ]:
def compute_minimal_cost(T, ls):
    """Returns (b_min, minimal error) over all segments of length ls."""
    raise NotImplementedError

# (T_0^(1e3), 50), (T_5^(1e4), 500), (T_10^(1e5), 5000), (T_20^(1e6), 1e4) -> list of (b_min, Err)
Q9 = [None, None, None, None]
print("Q9", Q9)

In [ ]:
def bu_apca(T, N):
    """Bottom-up APCA with N segments. Merge cost = Err of the merged segment,
    ties -> leftmost pair, exact comparisons. Returns the list of segments (b, e)."""
    raise NotImplementedError

# (T_0^(1000), 10), (T_5^(1e4), 20), (T_10^(2e4), 50), (T_20^(1e5), 100) -> list of (longest segment, total error)
Q10 = [None, None, None, None]
print("Q10", Q10)

**T12** — Complexity of a straightforward `BuAPCA`; strategy for $O(l\log l)$.

*Your answer:*

**T13** — Algorithm for the optimal APCA, justification, complexities; comparison with the enumeration of all segmentations.

*Your answer:*

In [ ]:
def optimal_apca(T, N):
    """Optimal APCA with N segments. Returns (segments, total error)."""
    raise NotImplementedError

# (T_0^(200), 5), (T_5^(500), 10), (T_10^(1000), 10), (T_20^(2000), 8) -> total error
Q11 = [None, None, None, None]
print("Q11", Q11)

**T14** — Is `BuAPCA` optimal? Smallest counter-example; comparison on the series of Q11.

*Your answer:*

**T15** — Prove $\mathsf{LB\_APCA}\le\mathsf{ED}$; complexity for the whole database.

*Your answer:*

In [ ]:
def lb_apca(Q, T_apca):
    """Lower bound of ED(Q, T) computed from Q and the APCA of T."""
    raise NotImplementedError

# database represented by bottom-up APCA with N segments
# #{T != NN_ED(Q) : LB_APCA(Q, T) < ED(Q, NN_ED(Q))} for (Q_2, N=4), (Q_2, N=8), (Q_3, N=4), (Q_3, N=8)
Q12 = [None, None, None, None]
print("Q12", Q12)

**T16** *(experiment)* — APCA ($N=8$) vs PAA ($N=16$): reconstruction error, number of candidates, tightness. Does a better approximation always lead to a better pruning? Explain.

In [ ]:
# T16: your experiments here

*Your answer:*

---
## 5. Lower bounding DTW and indexing with an R-tree *(at home)*

**T17** — Prove $\mathsf{LB\_Keogh}\le\mathsf{DTW}_w$.

*Your answer:*

**T18** — Complexity of the naive envelope; an $O(l)$ algorithm.

*Your answer:*

In [ ]:
def envelope(Q, w):
    """Returns (U, L), the envelope of Q for a band w."""
    raise NotImplementedError

def lb_keogh(Q, C, w):
    raise NotImplementedError

Q13 = [None, None, None, None]    # LB_Keogh(Q_j, T_j), w = 10, j = 0..3
Q14 = [None, None, None, None]    # #{T != NN : LB_Keogh(Q, T) < DTW_w(Q, NN)} for (Q_1, w=10), (Q_3, w=10), (Q_1, w=20), (Q_3, w=20)
print("Q13", Q13); print("Q14", Q14)

**T19** *(experiment)* — Tightness $\mathsf{LB\_Keogh}/\mathsf{DTW}_w$ as a function of $w$.

In [ ]:
# T19: your experiments here

*Your answer:*

**T20** — Prove $\mathsf{LB\_PAA}\le\mathsf{LB\_Keogh}$.

*Your answer:*

**T21** — Geometric interpretation in $\mathbb R^N$; adaptation of $\mathsf{MINDIST}$ (Section 5.3 of the subject) into $\mathsf{MINDIST}(Q,R)$ for an MBR $R$; proof that it lower-bounds $\mathsf{DTW}_w$ in the sub-tree; property (ii).

*Your answer:*

The cell below builds the **structure** of the R-tree (Sort-Tile-Recursive bulk loading, see Section 5.3 of the subject). **Do not modify it.** It returns nested tuples: a leaf is `("leaf", [series indices])`, an internal node is `("node", [children])`. Computing the MBRs is your job.

In [ ]:
# ---------------------------------------------------------------------------
# R-tree construction: Sort-Tile-Recursive (STR) bulk loading
# DO NOT MODIFY
# ---------------------------------------------------------------------------
def build_rtree(D, N, M=16):
    """Builds the *structure* of an R-tree indexing the PAA (order N) of the rows of D.

    A leaf is a pair ("leaf", [k_1, ..., k_m]) of at most M series indices.
    An internal node is a pair ("node", [child_1, ..., child_m]) of at most M nodes.
    Returns the root. The MBRs are NOT returned: computing them is your job.
    """
    n, l = D.shape
    # segment sums = (l/N) * PAA: exact integers, same order as the PAA
    pts = np.asarray(D, dtype=np.int64).reshape(n, N, l // N).sum(axis=2)

    def iroot_ceil(x, k):
        """smallest integer s >= 1 such that s**k >= x"""
        s = 1
        while s ** k < x:
            s += 1
        return s

    def str_pack(idx, X, dim):
        """STR: groups the items idx (coordinates X[idx]) into runs of at most M,
        tiling coordinate `dim`, then `dim + 1`, ... (k = N - dim remaining coordinates)."""
        k = N - dim
        idx = idx[np.lexsort((idx, X[idx, dim]))]        # sort by coordinate `dim` (ties: by index)
        if k == 1:
            return [idx[s:s + M] for s in range(0, len(idx), M)]
        P = -(-len(idx) // M)                            # number of pages to fill
        slab = M * iroot_ceil(P ** (k - 1), k)           # n * ceil(P^((k-1)/k)) items per slab
        groups = []
        for s in range(0, len(idx), slab):
            groups += str_pack(idx[s:s + slab], X, dim + 1)
        return groups

    # leaf level
    groups = str_pack(np.arange(n), pts, 0)
    level = [("leaf", [int(k) for k in g]) for g in groups]
    lo = np.array([pts[g].min(axis=0) for g in groups])
    hi = np.array([pts[g].max(axis=0) for g in groups])
    # upper levels: STR applied to the centres of the MBRs of the level below
    while len(level) > 1:
        centres = lo + hi                                # 2 x centre: exact integers, same order
        groups = str_pack(np.arange(len(level)), centres, 0)
        level = [("node", [level[i] for i in g]) for g in groups]
        lo = np.array([lo[g].min(axis=0) for g in groups])
        hi = np.array([hi[g].max(axis=0) for g in groups])
    return level[0]

In [ ]:
def lb_paa(Q, C_paa, w, N):
    raise NotImplementedError

def compute_mbrs(root, D_paa):
    """Attaches to every node of the tree returned by build_rtree the MBR (lower and upper corners)
    of the PAA of the series stored in its sub-tree."""
    raise NotImplementedError

def mindist(Q_paa_envelope, mbr):
    """Your adaptation of MINDIST to the PAA envelope of the query (T21)."""
    raise NotImplementedError

def rtree_nn_dtw(root, Q, D, w, N):
    """Exact best-first nearest neighbour search under DTW_w (adaptation of BestFirstNN).
    Suggested: also return the number of opened nodes, of LB_Keogh and of DTW_w computations."""
    raise NotImplementedError

root = build_rtree(D, 8)          # N = 8, M = 16

Q15 = [None, None, None, None]    # number of nodes of the tree (root, internal nodes and leaves) opened by your search, w = 10, N = 8, Q_0..Q_3
print("Q15", Q15)

**T22** — Description of your search algorithm and proof of exactness. Which nodes are opened? Why does Q15 not depend on how your priority queue breaks ties? Number of $\mathsf{LB\_Keogh}$ / $\mathsf{DTW}_w$ computations and link with Q14.

*Your answer:*

**T23** *(experiment)* — Brute force vs sequential scan with LB_Keogh vs index; influence of $N$ and of the node capacity.

In [ ]:
# T23: your experiments here

*Your answer:*

---
## Answers
Run this cell with **your** seed before handing in the notebook. Formats: plain numbers, or tuples `(k, distance)` for Q4/Q8, `(b_min, Err)` for Q9, `(longest, total error)` for Q10.

In [ ]:
ANSWERS = {f"Q{i}": globals().get(f"Q{i}") for i in range(1, 16)}
print("SEED =", SEED)
for q, v in ANSWERS.items():
    print(q, v)

### Self-check with the reference sheet (`SEED = 42` only)

In [ ]:
REF42 = {
    'Q1': [27, 28, 15, 22],
    'Q2': [181.6948, 118.2159, 100.3992, 240.7011],
    'Q3': [138.3172, 165.4277, 179.1431, 181.6948],
    'Q4': [[[252], 12.8841], [[723], 40.3361], [[204], 80.8641], [[969], 75.7166]],
    'Q5': [140, 100, 50, 25],
    'Q6': [47.1275, 83.5643, 993.4667, 5869.3572],
    'Q7': [181.6948, 161.6787, 117.601, 34104.7687],
    'Q8': [[[723], 37.8682], [[194], 14.1067], [[969], 62.5859], [[8], 63.8514]],
    'Q9': [[[728, 729], 26.88], [[7549, 7550, 7551, 7552], 5316.312], [[86477, 86478, 86479, 86480, 86481], 486854.8088], [[288523], 874928.5504]],
    'Q10': [[[184], 6025.1003], [[1230], 257269.8483], [[908], 458615.382], [[2320], 4958095.1802]],
    'Q11': [673.4105, 1064.4096, 6238.0738, 18741.1973],
    'Q12': [97, 43, 57, 14],
    'Q13': [168.2914, 70.342, 285.1631, 101.2225],
    'Q14': [55, 25, 120, 31],
    'Q15': [27, 47, 19, 45],
}

def check_reference(answers=None):
    """Compares your answers with the reference sheet of the subject (only meaningful with SEED = 42)."""
    if SEED != 42:
        print("The reference sheet is for SEED = 42."); return
    answers = ANSWERS if answers is None else answers
    def close(x, y):
        return abs(float(x) - float(y)) <= max(0.011, 1e-5 * abs(float(y)))
    for q, ref in REF42.items():
        got = answers.get(q) or [None] * 4
        marks = []
        for c, r, a in zip("abcd", ref, got):
            try:
                if isinstance(r, list):          # (index or set of accepted indices, value)
                    ks, v = r
                    ok = int(a[0]) in ks and close(a[1], v)
                else:
                    ok = close(a, r)
            except Exception:
                ok = False
            marks.append(f"{c}) {'ok' if ok else '--'}")
        print(f"{q:4s}", "  ".join(marks))

check_reference()